# Linear Regression — from scratch

Linear regression models a continuous target as a linear combination of the input features:

$$\hat{y} = w_1x_1 + w_2x_2 + \dots + w_dx_d + b = \mathbf{w}^\top\mathbf{x} + b$$

We pick $\mathbf{w}, b$ to minimise the **Mean Squared Error**:

$$J(\mathbf{w}, b) = \frac{1}{n}\sum_{i=1}^{n}(\hat{y}_i - y_i)^2$$

Two ways to solve it:

1. **Normal equation (closed form)** — prepend a column of ones to $X$ (call it $X_b$) and solve
   $$\boldsymbol{\theta} = (X_b^\top X_b)^{-1}X_b^\top y$$
2. **Gradient descent** — repeatedly step against the gradient
   $$\frac{\partial J}{\partial \mathbf{w}} = \frac{2}{n}X^\top(\hat{y}-y), \qquad \frac{\partial J}{\partial b} = \frac{2}{n}\sum_i(\hat{y}_i-y_i)$$

In [ ]:
import warnings
import numpy as np
import matplotlib.pyplot as plt

# NumPy 2.0 with Apple's Accelerate BLAS raises spurious "encountered in matmul" warnings on macOS
warnings.filterwarnings("ignore", message=".*encountered in matmul")

np.random.seed(42)
plt.rcParams["figure.figsize"] = (7, 5)

## Implementation

In [ ]:
class LinearRegression:
    """Ordinary least squares solved with the normal equation or batch gradient descent."""

    def __init__(self, method="gd", lr=0.01, n_iters=1000):
        assert method in ("gd", "normal")
        self.method = method
        self.lr = lr
        self.n_iters = n_iters

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        n, d = X.shape

        if self.method == "normal":
            Xb = np.c_[np.ones(n), X]                     # add bias column
            theta = np.linalg.pinv(Xb.T @ Xb) @ Xb.T @ y  # pinv is robust to singular X^T X
            self.b, self.w = theta[0], theta[1:]
            return self

        self.w = np.zeros(d)
        self.b = 0.0
        self.loss_history = []
        for _ in range(self.n_iters):
            error = X @ self.w + self.b - y
            self.w -= self.lr * (2 / n) * (X.T @ error)
            self.b -= self.lr * (2 / n) * error.sum()
            self.loss_history.append(np.mean(error ** 2))
        return self

    def predict(self, X):
        return np.asarray(X, dtype=float) @ self.w + self.b


def mean_squared_error(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)


def r2_score(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    return 1 - ss_res / ss_tot

## Simple linear regression (one feature)

Data generated from $y = 4 + 3x + \varepsilon$, $\varepsilon \sim \mathcal{N}(0, 1)$.

In [ ]:
X = 2 * np.random.rand(100, 1)
y = 4 + 3 * X[:, 0] + np.random.randn(100)

gd = LinearRegression(method="gd", lr=0.05, n_iters=1000).fit(X, y)
ne = LinearRegression(method="normal").fit(X, y)

print(f"Gradient descent : w = {gd.w[0]:.4f}, b = {gd.b:.4f}")
print(f"Normal equation  : w = {ne.w[0]:.4f}, b = {ne.b:.4f}")
print(f"True parameters  : w = 3.0000, b = 4.0000")
print(f"MSE = {mean_squared_error(y, gd.predict(X)):.4f},  R^2 = {r2_score(y, gd.predict(X)):.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

x_line = np.linspace(0, 2, 100).reshape(-1, 1)
axes[0].scatter(X, y, alpha=0.6, label="data")
axes[0].plot(x_line, gd.predict(x_line), "r-", lw=2, label="fitted line")
axes[0].set_xlabel("x"); axes[0].set_ylabel("y"); axes[0].legend()
axes[0].set_title("Linear regression fit")

axes[1].plot(gd.loss_history)
axes[1].set_xlabel("iteration"); axes[1].set_ylabel("MSE")
axes[1].set_title("Gradient descent convergence")
axes[1].set_yscale("log")
plt.tight_layout(); plt.show()

## Effect of the learning rate

Too small → slow convergence; too large → divergence.

In [ ]:
for lr in [0.001, 0.01, 0.1, 0.5]:
    model = LinearRegression(lr=lr, n_iters=200).fit(X, y)
    plt.plot(model.loss_history, label=f"lr = {lr}")
plt.yscale("log"); plt.xlabel("iteration"); plt.ylabel("MSE"); plt.legend()
plt.title("Learning rate comparison"); plt.show()

## Multiple linear regression

Three features with true relation $y = 2x_1 - 3x_2 + 0.5x_3 + 7 + \varepsilon$.

Gradient descent converges much faster when features are on the same scale, so we **standardise** them
(the normal equation does not need this).

In [ ]:
n = 500
X_multi = np.random.randn(n, 3) * [1, 5, 20] + [0, 10, -5]
true_w, true_b = np.array([2.0, -3.0, 0.5]), 7.0
y_multi = X_multi @ true_w + true_b + np.random.randn(n)

# standardise
mu, sigma = X_multi.mean(axis=0), X_multi.std(axis=0)
X_std = (X_multi - mu) / sigma

model = LinearRegression(lr=0.1, n_iters=500).fit(X_std, y_multi)

# convert weights back to the original feature scale
w_orig = model.w / sigma
b_orig = model.b - np.sum(model.w * mu / sigma)
print("Recovered w:", np.round(w_orig, 3), " b:", round(b_orig, 3))
print("True      w:", true_w, " b:", true_b)

ne = LinearRegression(method="normal").fit(X_multi, y_multi)
print("Normal eq w:", np.round(ne.w, 3), " b:", round(ne.b, 3))
print("R^2:", round(r2_score(y_multi, ne.predict(X_multi)), 4))

In [ ]:
pred = ne.predict(X_multi)
plt.scatter(y_multi, pred, alpha=0.5)
lims = [y_multi.min(), y_multi.max()]
plt.plot(lims, lims, "r--", label="perfect prediction")
plt.xlabel("actual y"); plt.ylabel("predicted y"); plt.legend()
plt.title("Predicted vs actual (multiple regression)"); plt.show()

## Summary
* Linear regression minimises squared error; the problem is convex so GD reaches the global optimum.
* The normal equation is exact but costs $O(d^3)$ — fine for few features, slow for very many.
* Gradient descent scales to large data but needs a tuned learning rate and scaled features.